# Clase 9 · Las tres preguntas que dejó el χ²

**Estadística Descriptiva e Inferencial** · Módulo 3 · Sesión 9 de 14

---

## De dónde venimos

La clase pasada aprendiste a responder **«¿hay relación?»** con el χ². Pero ese número
dejó tres preguntas abiertas, y las tres tienen respuesta.

| Pregunta que quedó | Herramienta de hoy |
|---|---|
| 1 · El χ² dice que hay relación… ¿pero **dónde**? | **Residuos** |
| 2 · ¿Y si mi tabla tiene **muy pocos casos**? | **Fisher exacto** |
| 3 · ¿Y si la relación que veo es una **ilusión**? | **Paradoja de Simpson** |

La tercera es la más incómoda del curso: vas a ver una tabla donde la conclusión
**se invierte** al mirar los datos con más detalle.

## El laboratorio

| Bloque | Min | Qué haces |
|---|---|---|
| 1 | 12 | Encuentras qué casilla causa la diferencia |
| 2 | 12 | Ves al χ² fallar con pocos datos, y lo arreglas con Fisher |
| 3 | 15 | **Descubres una conclusión que se da vuelta** |
| 4 | 6 | Una prueba parecida pero distinta, para no confundirlas |

> Como en la clase anterior: tablas pequeñas, código explícito y todo verificable a mano.
> No hay que descargar nada.

---
## Celda 0 · Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

NAVY, BLUE, MAG, GREEN = "#0A2559", "#1A56E8", "#E6115E", "#12B886"
plt.rcParams.update({
    "figure.figsize": (8, 4), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None:
        print(f"[ ] {nombre}: todavia no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

print("Listo. Empezamos.")

---
# Bloque 1 · ¿Dónde está la diferencia?  ·  12 min

Tenemos **600 solicitudes**, 200 por cada canal:

| | Rechazada | Aprobada | Total |
|---|---|---|---|
| **Agencia** | 80 | 120 | 200 |
| **App** | 40 | 160 | 200 |
| **Web** | 60 | 140 | 200 |
| **Total** | 180 | 420 | 600 |

En total se rechaza el 30 %, así que **esperaríamos 60 rechazos en cada canal** si el canal
no importara.

El χ² te va a decir que hay relación. Pero con tres canales, **¿cuál es el responsable?**

### Ejercicio 1.1 — El χ² y la tabla esperada

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
observado = np.array([[80, 120],
                      [40, 160],
                      [60, 140]])

filas    = ["agencia", "app", "web"]
columnas = ["rechazada", "aprobada"]

resultado = stats.chi2_contingency(observado)
esperado  = resultado.expected_freq

print(f"chi2 = {resultado.statistic:.4f}")
print(f"p    = {resultado.pvalue:.6f}")
print(f"gl   = {resultado.dof}     (3 filas - 1) x (2 columnas - 1) = 2")
print()
print("Tabla ESPERADA:")
print(pd.DataFrame(esperado, index=filas, columns=columnas))
print()
print("Los tres canales esperan lo mismo, porque los tres tienen 200 solicitudes.")
print()
print("Porcentaje de rechazo real de cada canal:")
for i in range(3):
    print(f"  {filas[i]:8}: {observado[i,0]}/200 = {100*observado[i,0]/200:.0f} %")
print()
print("p < 0.05, asi que hay relacion. Pero, ¿cual canal es el responsable?")
print("El chi2 es UN SOLO numero para toda la tabla: no lo dice.")

In [ ]:
# ── VERIFICACIÓN 1.1 ─────────────────────────────────────────────────────
r = [check("chi cuadrado", resultado.statistic, 19.0476),
     check("grados de libertad", resultado.dof, 2),
     check("esperado en cada celda de 'rechazada'", esperado[0, 0], 60)]
print()
print("1.1 OK" if all(r) else "Revisa 1.1")

### Ejercicio 1.2 — Los residuos: la diferencia, casilla por casilla

Un **residuo** es simplemente la diferencia entre lo observado y lo esperado, puesta en una
escala comparable:

$$\text{residuo} = \frac{\text{observado} - \text{esperado}}{\sqrt{\text{esperado}}}$$

Se lee así:
- **positivo** → hay **más** casos de los esperados en esa casilla
- **negativo** → hay **menos**
- **cerca de cero** → esa casilla es justo lo que se esperaba

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
residuos = np.zeros((3, 2))

for i in range(3):
    for j in range(2):
        residuos[i, j] = (observado[i, j] - esperado[i, j]) / np.sqrt(esperado[i, j])

print("Residuos:")
print(pd.DataFrame(residuos, index=filas, columns=columnas).round(3))
print()
print("Como se lee esto:")
print(f"  agencia x rechazada = {residuos[0,0]:+.3f}  ->  MAS rechazos de los esperados")
print(f"  app     x rechazada = {residuos[1,0]:+.3f}  ->  MENOS rechazos de los esperados")
print(f"  web     x rechazada = {residuos[2,0]:+.3f}  ->  EXACTAMENTE lo esperado")
print()
print("Web tiene residuo CERO: sus 60 rechazos son justo los 60 que esperabamos.")
print("Web no aporta nada al chi2. Toda la diferencia viene de agencia y app.")
print()
# Y una comprobacion bonita
print(f"Suma de los residuos al cuadrado = {(residuos**2).sum():.4f}")
print(f"El chi cuadrado                  = {resultado.statistic:.4f}")
print("Son el mismo numero. El chi2 ES la suma de los residuos al cuadrado.")

In [ ]:
# ── VERIFICACIÓN 1.2 ─────────────────────────────────────────────────────
r = [check("residuo de agencia-rechazada", residuos[0, 0], 2.5820),
     check("residuo de app-rechazada", residuos[1, 0], -2.5820),
     check("residuo de web-rechazada", residuos[2, 0], 0.0),
     check("la suma de residuos² es el chi2", (residuos**2).sum(), resultado.statistic)]
print()
print("1.2 OK" if all(r) else "Revisa 1.2")

### Ejercicio 1.3 — El residuo ajustado y la regla del 2

El residuo simple tiene un problema pequeño: su escala depende de los totales de la fila y
la columna. El **residuo ajustado** lo corrige:

$$\text{ajustado} = \frac{\text{observado} - \text{esperado}}
{\sqrt{\text{esperado} \times (1 - \frac{\text{total fila}}{n}) \times (1 - \frac{\text{total col}}{n})}}$$

**La regla práctica:** si el valor absoluto del residuo ajustado **supera 2**, esa casilla se
aparta claramente de lo esperado. Es la casilla que hay que mirar.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
n_total   = observado.sum()
tot_filas = observado.sum(axis=1)
tot_cols  = observado.sum(axis=0)

ajustados = np.zeros((3, 2))

for i in range(3):
    for j in range(2):
        denominador = esperado[i, j] * (1 - tot_filas[i]/n_total) * (1 - tot_cols[j]/n_total)
        ajustados[i, j] = (observado[i, j] - esperado[i, j]) / np.sqrt(denominador)

print("Residuos AJUSTADOS:")
print(pd.DataFrame(ajustados, index=filas, columns=columnas).round(3))
print()
print("Aplicando la regla del 2:")
for i in range(3):
    for j in range(2):
        v = ajustados[i, j]
        marca = "  <-- DESTACA" if abs(v) > 2 else ""
        print(f"  {filas[i]:8} x {columnas[j]:10}: {v:+7.3f}{marca}")
print()
print("CONCLUSION que ahora si puedes escribir en un informe:")
print("  'Hay relacion entre canal y resultado (chi2 = 19.05, gl = 2, p < 0.001).")
print("   El canal AGENCIA concentra mas rechazos de los esperados y APP menos;")
print("   WEB se comporta exactamente como se esperaria.'")
print()
print("Fijate en la diferencia con la clase pasada: antes solo podias decir")
print("'hay relacion'. Ahora puedes decir DONDE esta.")

In [ ]:
# ── VERIFICACIÓN 1.3 ────────────────────────────────────────────────────
r = [check("residuo ajustado de agencia-rechazada", ajustados[0, 0], 3.7796),
     check("residuo ajustado de app-rechazada", ajustados[1, 0], -3.7796),
     check("residuo ajustado de web-rechazada", ajustados[2, 0], 0.0),
     check_bool("agencia y app superan la regla del 2",
                abs(ajustados[0, 0]) > 2 and abs(ajustados[1, 0]) > 2),
     check_bool("web NO la supera", abs(ajustados[2, 0]) < 2)]
print()
print("Bloque 1 COMPLETO" if all(r) else "Revisa 1.3")

---
# Bloque 2 · Cuando hay muy pocos casos  ·  12 min

Probaste un **control antifraude nuevo** contra el que ya tenías. Solo pudiste hacer la
prueba con 20 casos, 10 con cada control:

| | Fraude que escapó | Fraude detectado | Total |
|---|---|---|---|
| **Control nuevo** | 1 | 9 | 10 |
| **Control viejo** | 6 | 4 | 10 |
| **Total** | 7 | 13 | 20 |

A ojo, el control nuevo se ve mucho mejor: dejó escapar 1 de 10 contra 6 de 10.

Pero con solo 20 casos, **el χ² deja de ser confiable.** Vamos a ver por qué y qué usar en
su lugar.

### Ejercicio 2.1 — Comprueba que el χ² no aplica

La regla que vimos la clase pasada: si alguna frecuencia **esperada** es menor que 5, el χ²
no es de fiar.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
control = np.array([[1, 9],
                    [6, 4]])

res_control  = stats.chi2_contingency(control)
esp_control  = res_control.expected_freq
esperada_min = esp_control.min()

print("Tabla ESPERADA:")
print(pd.DataFrame(esp_control, index=["nuevo", "viejo"],
                   columns=["escapo", "detectado"]))
print()
print(f"La frecuencia esperada mas chica es {esperada_min:.1f}")
print()
if esperada_min < 5:
    print("Es MENOR que 5, asi que el chi cuadrado NO es confiable aqui.")
    print("El chi2 usa una aproximacion que necesita suficientes casos por casilla,")
    print("y con estos numeros esa aproximacion se rompe.")
else:
    print("Es mayor que 5: el chi2 se puede usar.")

In [ ]:
# ── VERIFICACIÓN 2.1 ─────────────────────────────────────────────────────
r = [check("frecuencia esperada mínima", esperada_min, 3.5),
     check_bool("es menor que 5, así que el chi2 no aplica", esperada_min < 5)]
print()
print("2.1 OK" if all(r) else "Revisa 2.1")

### Ejercicio 2.2 — Fisher exacto, y por qué importa

La **prueba exacta de Fisher** no usa ninguna aproximación: calcula directamente la
probabilidad de obtener una tabla así de extrema si no hubiera relación.

Es la prueba que Fisher inventó para el experimento del té que vimos la clase pasada.

Compara los tres resultados y fíjate bien en si la **conclusión** cambia.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
p_chi2_sin_yates = stats.chi2_contingency(control, correction=False).pvalue
p_chi2_con_yates = stats.chi2_contingency(control).pvalue
p_fisher         = stats.fisher_exact(control).pvalue

print(f"{'Metodo':28}{'p-valor':>10}   Conclusion a 0.05")
print("-" * 62)
print(f"{'chi2 sin correccion':28}{p_chi2_sin_yates:>10.4f}   "
      f"{'SIGNIFICATIVO' if p_chi2_sin_yates < 0.05 else 'no significativo'}")
print(f"{'chi2 con correccion (Yates)':28}{p_chi2_con_yates:>10.4f}   "
      f"{'SIGNIFICATIVO' if p_chi2_con_yates < 0.05 else 'no significativo'}")
print(f"{'FISHER EXACTO':28}{p_fisher:>10.4f}   "
      f"{'SIGNIFICATIVO' if p_fisher < 0.05 else 'no significativo'}")
print("-" * 62)
print()
print("MIRA LA ULTIMA COLUMNA: la conclusion CAMBIA segun el metodo.")
print()
print("El chi2 sin correccion dice que si hay diferencia (p = 0.019).")
print("Fisher, que es el correcto aqui, dice que no alcanza (p = 0.057).")
print()
print("Y Fisher tiene razon: con 20 casos no hay evidencia suficiente,")
print("aunque 1 contra 6 parezca una diferencia enorme.")
print()
print("Lo que hay que hacer: mas pruebas. La diferencia probablemente existe,")
print("pero 20 casos no bastan para demostrarla.")

In [ ]:
# ── VERIFICACIÓN 2.2 ─────────────────────────────────────────────────────
r = [check("p del chi2 sin corrección", p_chi2_sin_yates, 0.0191, tol=1e-3),
     check("p de Fisher exacto", p_fisher, 0.0573, tol=1e-3),
     check_bool("el chi2 sin corrección dice significativo", p_chi2_sin_yates < 0.05),
     check_bool("pero Fisher dice que NO alcanza", p_fisher >= 0.05)]
print()
print("La leccion: con pocos casos, el chi2 es demasiado OPTIMISTA.")
print("Te hace ver diferencias que los datos no sostienen.")
print()
print("Bloque 2 COMPLETO" if all(r) else "Revisa 2.2")

---
# Bloque 3 · La conclusión que se da vuelta  ·  15 min

**Este es el bloque más importante del día, y el más incómodo.**

Un banco compara la tasa de aprobación de dos canales. Tiene 600 solicitudes, 300 por
canal, y quiere saber cuál aprueba más.

Vamos a mirarlo primero **en total**, y después **separado por segmento de cliente**.

Presta atención a lo que pasa.

In [ ]:
# ── DEMOSTRACIÓN: los datos ──────────────────────────────────────────────
datos = pd.DataFrame([
    {"canal": "app",     "segmento": "facil",   "aprobadas":  80, "total": 100},
    {"canal": "app",     "segmento": "dificil", "aprobadas":  20, "total": 200},
    {"canal": "agencia", "segmento": "facil",   "aprobadas": 140, "total": 200},
    {"canal": "agencia", "segmento": "dificil", "aprobadas":   5, "total": 100},
])
datos["tasa_%"] = (100 * datos["aprobadas"] / datos["total"]).round(1)
print(datos.to_string(index=False))
print()
print("'facil' y 'dificil' son segmentos de riesgo del cliente:")
print("  facil   = cliente con buen historial, se aprueba mucho")
print("  dificil = cliente sin historial o con problemas, se aprueba poco")

### Ejercicio 3.1 — Primero, la mirada TOTAL

Suma los dos segmentos de cada canal y calcula la tasa de aprobación general.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
app     = datos[datos.canal == "app"]
agencia = datos[datos.canal == "agencia"]

total_app     = app["aprobadas"].sum()
solicit_app   = app["total"].sum()
tasa_app      = 100 * total_app / solicit_app

total_agencia   = agencia["aprobadas"].sum()
solicit_agencia = agencia["total"].sum()
tasa_agencia    = 100 * total_agencia / solicit_agencia

print("MIRADA TOTAL (sin separar por segmento)")
print("-" * 48)
print(f"  app     : {total_app:3} de {solicit_app} aprobadas  =  {tasa_app:.1f} %")
print(f"  agencia : {total_agencia:3} de {solicit_agencia} aprobadas  =  {tasa_agencia:.1f} %")
print("-" * 48)
print()
print(f"AGENCIA aprueba mas: {tasa_agencia:.1f} % contra {tasa_app:.1f} %.")
print(f"Son {tasa_agencia - tasa_app:.1f} puntos de diferencia. Parece claro.")
print()
# Y ademas sale significativo
tabla_total = np.array([[total_app, solicit_app - total_app],
                        [total_agencia, solicit_agencia - total_agencia]])
res_total = stats.chi2_contingency(tabla_total, correction=False)
print(f"Y el chi2 lo confirma: chi2 = {res_total.statistic:.2f}, p = {res_total.pvalue:.5f}")
print("Con este analisis cerrarias el informe: 'agencia aprueba mas que app'.")

In [ ]:
# ── VERIFICACIÓN 3.1 ─────────────────────────────────────────────────────
r = [check("tasa de aprobación de app (%)", tasa_app, 33.3333, tol=0.01),
     check("tasa de aprobación de agencia (%)", tasa_agencia, 48.3333, tol=0.01),
     check_bool("con esta mirada, agencia gana", tasa_agencia > tasa_app)]
print()
print("3.1 OK" if all(r) else "Revisa 3.1")

### Ejercicio 3.2 — Ahora, separado por segmento

Mismos datos. Solo que ahora comparamos canal contra canal **dentro de cada segmento**.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
print("MIRADA POR SEGMENTO")
print("-" * 62)
for seg in ["facil", "dificil"]:
    fila_app = datos[(datos.canal == "app") & (datos.segmento == seg)].iloc[0]
    fila_agn = datos[(datos.canal == "agencia") & (datos.segmento == seg)].iloc[0]
    t_app     = 100 * fila_app["aprobadas"] / fila_app["total"]
    t_agencia = 100 * fila_agn["aprobadas"] / fila_agn["total"]
    gana = "APP" if t_app > t_agencia else "AGENCIA"
    print(f"  segmento {seg:8}:  app {t_app:5.1f} %   agencia {t_agencia:5.1f} %   ->  gana {gana}")
print("-" * 62)
print()
print("=" * 62)
print("APP GANA EN LOS DOS SEGMENTOS.")
print("Pero en el total ganaba AGENCIA.")
print("=" * 62)
print()
print("Los datos son exactamente los mismos. No hay ningun error de calculo.")
print("Esto se llama PARADOJA DE SIMPSON.")

In [ ]:
# ── VERIFICACIÓN 3.2 ─────────────────────────────────────────────────────
def tasa(canal, seg):
    f = datos[(datos.canal == canal) & (datos.segmento == seg)].iloc[0]
    return 100 * f["aprobadas"] / f["total"]

r = [check("tasa de app en 'facil' (%)", tasa("app", "facil"), 80.0),
     check("tasa de agencia en 'facil' (%)", tasa("agencia", "facil"), 70.0),
     check("tasa de app en 'dificil' (%)", tasa("app", "dificil"), 10.0),
     check("tasa de agencia en 'dificil' (%)", tasa("agencia", "dificil"), 5.0),
     check_bool("app gana en 'facil'", tasa("app", "facil") > tasa("agencia", "facil")),
     check_bool("app gana en 'dificil'", tasa("app", "dificil") > tasa("agencia", "dificil")),
     check_bool("pero agencia gana en el total", tasa_agencia > tasa_app)]
print()
print("3.2 OK" if all(r) else "Revisa 3.2")

### Ejercicio 3.3 — ¿Por qué pasa esto?

No es magia ni un error. La explicación está en **cómo se reparten los volúmenes**.

Calcula qué porcentaje de las solicitudes de cada canal viene de cada segmento.

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
print("COMO SE REPARTEN LOS VOLUMENES")
print("-" * 56)
for canal in ["app", "agencia"]:
    sub = datos[datos.canal == canal]
    tot = sub["total"].sum()
    pct_facil   = 100 * sub[sub.segmento == "facil"]["total"].iloc[0] / tot
    pct_dificil = 100 * sub[sub.segmento == "dificil"]["total"].iloc[0] / tot
    print(f"  {canal:8}: {pct_facil:.0f} % del segmento facil, {pct_dificil:.0f} % del dificil")
print("-" * 56)
print()
print("AHI ESTA LA EXPLICACION:")
print()
print("  App atiende sobre todo clientes DIFICILES (67 % de su volumen).")
print("  Agencia atiende sobre todo clientes FACILES (67 % de su volumen).")
print()
print("Como el segmento dificil se aprueba poco POR NATURALEZA, app arrastra")
print("una tasa total baja aunque haga mejor trabajo en cada segmento.")
print()
print("El segmento es lo que se llama una VARIABLE DE CONFUSION: influye en el")
print("resultado Y esta repartida de forma desigual entre los canales.")
print()
print("La pregunta clave para decidir cual mirada es la correcta:")
print("  ¿el canal ELIGE que clientes atiende, o le vienen dados?")
print()
print("  Si le vienen dados -> hay que comparar DENTRO de cada segmento.")
print("     App hace mejor trabajo. La mirada total es enganosa.")
print()
print("  Si el canal elige a quien atender -> el total puede ser relevante,")
print("     porque esa eleccion es parte de su desempeno.")

In [ ]:
# ── VERIFICACIÓN 3.3 ─────────────────────────────────────────────────────
vol_app_dif = 100 * datos[(datos.canal=="app") & (datos.segmento=="dificil")]["total"].iloc[0] / 300
vol_agn_dif = 100 * datos[(datos.canal=="agencia") & (datos.segmento=="dificil")]["total"].iloc[0] / 300
r = [check("% del volumen de app que es 'dificil'", vol_app_dif, 66.6667, tol=0.01),
     check("% del volumen de agencia que es 'dificil'", vol_agn_dif, 33.3333, tol=0.01),
     check_bool("app concentra más casos difíciles que agencia", vol_app_dif > vol_agn_dif)]
print()
print("Lo que hay que llevarse del bloque:")
print("  antes de comparar dos grupos, pregunta si hay alguna variable que")
print("  este repartida de forma desigual entre ellos. Si la hay, el total puede mentir.")
print()
print("Bloque 3 COMPLETO" if all(r) else "Revisa 3.3")

---
# Bloque 4 · Una prueba parecida, pero distinta  ·  6 min

Existe otra prueba que también se llama **χ²** y que usa la misma fórmula, pero responde
una pregunta diferente. Conviene conocerla para no confundirlas.

| | χ² de independencia (Clase 8) | χ² de bondad de ajuste |
|---|---|---|
| Compara | **dos** variables entre sí | **una** variable contra una expectativa |
| La pregunta | «¿tienen que ver una con otra?» | «¿se reparte como yo esperaba?» |
| Los esperados salen de | los totales de la tabla | **tú los propones** |
| En Python | `chi2_contingency` | `chisquare` |

**Ejemplo:** el equipo de fraude debería repartir sus revisiones por igual entre los cinco
días de la semana. ¿Lo hace?

In [ ]:
# ── SOLUCIÓN ─────────────────────────────────────────────────────────────
revisiones = np.array([120, 95, 100, 105, 80])
dias = ["lunes", "martes", "miercoles", "jueves", "viernes"]

esperado_por_dia = revisiones.sum() / len(revisiones)
res_ajuste = stats.chisquare(revisiones)

print(f"Total de revisiones: {revisiones.sum()}")
print(f"Si se repartieran por igual: {esperado_por_dia:.0f} por dia")
print()
for i in range(5):
    dif = revisiones[i] - esperado_por_dia
    print(f"  {dias[i]:10}: {revisiones[i]:3}   ({dif:+.0f} respecto a lo esperado)")
print()
print(f"chi2 = {res_ajuste.statistic:.4f}   p = {res_ajuste.pvalue:.4f}")
print()
if res_ajuste.pvalue < 0.05:
    print("p < 0.05: el reparto NO es uniforme. Los lunes se revisa de mas")
    print("y los viernes de menos, mas de lo que explicaria el azar.")
else:
    print("p >= 0.05: no hay evidencia de que el reparto sea desigual.")
    print("Las diferencias que ves caben dentro de la variacion normal.")
print()
print("Fijate en la diferencia clave con la Clase 8: aqui NO hay dos variables.")
print("Hay UNA (el dia) y una expectativa que propusimos nosotros (reparto igual).")

In [ ]:
# ── VERIFICACIÓN 4 ──────────────────────────────────────────────────────
r = [check("esperado por día", esperado_por_dia, 100),
     check("chi cuadrado de bondad de ajuste", res_ajuste.statistic, 8.5, tol=0.01),
     check_bool("con p >= 0.05, no hay evidencia de reparto desigual",
                res_ajuste.pvalue >= 0.05)]
print()
print("Un resultado util: aunque el lunes tenga 120 y el viernes 80, esa")
print("diferencia cabe dentro del azar con 500 revisiones. No hay que")
print("reorganizar nada todavia.")
print()
print("LABORATORIO COMPLETO" if all(r) else "Revisa el bloque 4")

---
# Cierre

### Las tres respuestas de hoy

| Pregunta | Respuesta | En Python |
|---|---|---|
| ¿Dónde está la relación? | Residuos ajustados, regla del 2 | a mano, 3 líneas |
| ¿Y si hay pocos casos? | Fisher exacto | `stats.fisher_exact` |
| ¿Y si la relación es ilusoria? | Separar por la variable de confusión | mirar por grupos |

### Checklist de salida

- [ ] Sé calcular residuos y usar la regla del 2.
- [ ] Sé que si una esperada es menor que 5, va Fisher y no χ².
- [ ] Sé que con pocos casos el χ² es demasiado optimista.
- [ ] Sé reconocer una posible paradoja de Simpson.
- [ ] Antes de comparar dos grupos, pregunto qué variable puede estar desbalanceada.

### Los números del día

| | |
|---|---|
| Residuo de web | **0.000** — exactamente lo esperado |
| Suma de residuos² | **19.0476** = el χ² exacto |
| Control nuevo: χ² sin corrección | p = **0.019** → «significativo» |
| Control nuevo: Fisher | p = **0.057** → **no alcanza** |
| Simpson: total | agencia 48.3 % vs app 33.3 % |
| Simpson: por segmento | app gana en **los dos** |

### El error que evita la clase de hoy

Un informe que dice **«agencia aprueba más que app»** con p < 0.001 puede ser
completamente correcto en el cálculo y completamente equivocado en la conclusión.

La estadística no te avisa. Lo único que te protege es preguntar, antes de comparar:
**¿hay alguna variable repartida de forma desigual entre los grupos?**

### Reto para la próxima clase

Toma una comparación entre dos grupos que exista en tu trabajo. Busca una tercera variable
que sospeches desbalanceada entre ellos: antigüedad del cliente, monto, región, tipo de
producto.

Separa la comparación por esa variable. **¿Se mantiene la conclusión, o cambia?**

---
*Estadística Descriptiva e Inferencial · Módulo 3 · Clase 9*